# Revision experiments — WiDS ICU mortality

Three analyses requested in the major revision, on the same code and CPU stack as
the paper's runs:

* **A.** Five-seed runs of both regimes with the privacy attacks switched on and
  per-record test predictions saved (reliability diagrams, complete-case
  APACHE IVa comparison, per-feature reconstruction error, multi-seed attacks).
* **B.** Record-level DP sweep on the embedding channel over five seeds.
* **C.** Pooling-threshold sensitivity: every hospital with ≥2 training records
  as its own client, plus thresholds 50 and 100 (200 is the paper's setting).

Attach `fedlgbm-src` and the **WiDS Datathon 2020** competition.
Run with **Save Version → Save & Run All (Commit)**. Expect roughly 3 hours.

In [ ]:
import sys, time, json, shutil
from pathlib import Path

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/outputs')

hits = sorted(INPUT.glob('*/**/fedlgbm/__init__.py'))
assert hits, 'fedlgbm not found — attach the fedlgbm-src dataset'
sys.path.insert(0, str(hits[0].parent.parent))

train_files = sorted(INPUT.glob('**/training_v2.csv'))
assert train_files, 'attach the WiDS Datathon 2020 competition'
DATA = train_files[0].parent
print('source:', hits[0].parent.parent)
print('data:', train_files[0])

# CPU on purpose: the paper's runs were CPU, and results are not bit-identical
# across devices, so a GPU here would make these numbers incomparable with them.
DEVICE = 'cpu'
print('device =', DEVICE)


In [ ]:
from fedlgbm import ExperimentConfig, setup_logging
from fedlgbm.ablations import run_multi_seed, sweep_privacy_utility
import traceback

setup_logging()

# Fail in seconds, not hours, if Kaggle mounted a stale fedlgbm-src version:
# the old code would ignore save_predictions and skip per-feature errors.
import dataclasses, inspect
from fedlgbm import attacks as _attacks
assert 'save_predictions' in {f.name for f in dataclasses.fields(ExperimentConfig)}, 'stale fedlgbm-src'
assert 'feature_names' in inspect.signature(_attacks.evaluate_privacy).parameters, 'stale fedlgbm-src'
print('fedlgbm-src version check passed')

def build(split_mode: str, name: str) -> ExperimentConfig:
    # Identical to the paper's multi-seed configuration (kaggle_wids_multiseed).
    cfg = ExperimentConfig()
    cfg.name = name
    cfg.output_dir = OUT
    cfg.device = DEVICE
    cfg.skip_privacy_attacks = True
    cfg.save_predictions = False

    cfg.data.domain = 'wids'
    cfg.data.data_dir = DATA
    cfg.data.file_glob = 'training_v2.csv'
    cfg.data.max_rows = None
    cfg.data.max_per_class = None
    cfg.data.split_mode = split_mode

    cfg.partition.scheme = 'natural'
    cfg.partition.min_samples_per_client = 200

    cfg.fed.n_rounds = 30
    cfg.fed.local_epochs = 2
    cfg.mlp.hidden_dims = (256, 128)
    cfg.mlp.embedding_dim = 64

    cfg.lgbm.class_weight = None
    cfg.lgbm.num_leaves = 255
    cfg.lgbm.n_estimators = 1000
    return cfg

SEEDS = (42, 43, 44, 45, 46)
SKIP = {'lstm', 'isolation_forest', 'stacked_mlp_lgbm', 'centralized_mlp_lgbm'}
ONLY_FED = SKIP | {'centralized_lgbm', 'fedavg_mlp_only', 'random_forest', 'logistic_regression'}
WITH_CENTRAL = SKIP | {'fedavg_mlp_only', 'random_forest', 'logistic_regression'}
REFERENCE = 'fed_hybrid_mlp_lgbm'
DONE = {}

def part(label, fn):
    # One failed analysis must not cost the others.
    t0 = time.time()
    print('=' * 70)
    print(label, flush=True)
    print('=' * 70)
    try:
        DONE[label] = fn()
        print(f'{label}: done in {(time.time() - t0) / 60:.1f} min', flush=True)
    except Exception:
        traceback.print_exc()
        DONE[label] = 'FAILED'

In [ ]:
# C. Pooling-threshold sensitivity. A threshold of 2 makes every hospital with at
# least two training records its own client (a one-record site cannot form a batch).
def run_c():
    out = {}
    for thr in (2, 50, 100):
        cfg = build('random', f'rev_C_within_site_min{thr}')
        cfg.partition.min_samples_per_client = thr
        out[f'within_min{thr}'] = run_multi_seed(cfg, seeds=SEEDS, skip=WITH_CENTRAL,
                                                 reference=REFERENCE)
    cfg = build('site_holdout', 'rev_C_cross_site_min2')
    cfg.partition.min_samples_per_client = 2
    out['cross_min2'] = run_multi_seed(cfg, seeds=SEEDS, skip=WITH_CENTRAL, reference=REFERENCE)
    return out

part('C_pooling_sensitivity', run_c)

In [ ]:
# A. Five seeds, both regimes, attacks on, per-record predictions saved.
def run_a():
    out = {}
    for mode, name in (('random', 'rev_A_within_site'), ('site_holdout', 'rev_A_cross_site')):
        cfg = build(mode, name)
        cfg.skip_privacy_attacks = False
        cfg.save_predictions = True
        out[name] = run_multi_seed(cfg, seeds=SEEDS, skip=SKIP, reference=REFERENCE)
    return out

part('A_multiseed_attacks_predictions', run_a)

In [ ]:
# B. Record-level DP on the embedding channel, every budget on all five seeds.
def run_b():
    cfg = build('random', 'rev_B_dp_embeddings')
    cfg.skip_privacy_attacks = False
    return sweep_privacy_utility(cfg, epsilons=(0.5, 1.0, 3.0, 5.0, 10.0),
                                 seeds=SEEDS, skip=ONLY_FED, channels='embeddings')

part('B_dp_sweep_multiseed', run_b)

In [ ]:
for k, v in DONE.items():
    print(k, 'FAILED' if v == 'FAILED' else 'ok')
shutil.make_archive('/kaggle/working/revision_outputs', 'zip', OUT)
print('archived /kaggle/working/revision_outputs.zip')